# Amazon Product Reviews — Feature Engineering (v4 — Simplified)

**Why simplified?**
The 28-feature version showed extreme overfitting (Train RMSE ~0.04, Test RMSE ~1.7).
Root cause: too many features derived from user/product averages created
near-perfect memorisation of training patterns that didn't generalise.

**6 safe features kept:**
| Feature | Type | Leakage risk |
|---------|------|-------------|
| `user_avg_rating` | User static | LOO — safe |
| `user_rating_count` | User static | Count only — safe |
| `product_avg_rating` | Product static | LOO — safe |
| `product_rating_count` | Product static | Count only — safe |
| `product_rating_std` | Product static | No direct target signal |
| `days_since_review` | Temporal | Pure timestamp — safe |

**What was dropped and why:**
- Bayesian avgs, diffs — derived from avg, amplify same signal
- user_generosity — derived from user_avg
- All 90-day window features — future leakage within train
- burstiness, ratings_per_month — computed on full train history
- entropy, cv, min, max — too correlated with avg

**Output files**
```
amazon/data/
  train_features.csv
  val_features.csv
  test_features.csv
```

**Known limitations:**
- LOO leakage for std (acceptable — small effect)
- Temporal features use TRAIN_MAX_DATE as anchor (minor, train-only)


## Roadmap
```
STEP 1 — Imports & paths
STEP 2 — Load splits
STEP 3 — Parse timestamps
STEP 4 — User features   (LOO avg + count)
STEP 5 — Product features (LOO avg + count + std)
STEP 6 — Apply to val & test
STEP 7 — Cold start handling
STEP 8 — Final check
STEP 9 — Save CSVs
```


---
## Step 1 — Imports & Paths

In [ ]:
import os
import numpy  as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


In [ ]:
BASE_DIR = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\amazon"
DATA_DIR = os.path.join(BASE_DIR, "data")
print(f"Data directory: {DATA_DIR}")


---
## Step 2 — Load Splits

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"Train : {df_train.shape}")
print(f"Val   : {df_val.shape}")
print(f"Test  : {df_test.shape}")
print(f"Columns: {list(df_train.columns)}")


---
## Step 3 — Parse Timestamps

Convert Unix timestamp to datetime.
Compute days_since_review relative to TRAIN_MAX_DATE.


In [ ]:
for df_ in [df_train, df_val, df_test]:
    df_['datetime'] = pd.to_datetime(df_['timestamp'], unit='s', errors='coerce')

TRAIN_MAX_DATE = df_train['datetime'].max()
print(f"Train date range : {df_train['datetime'].min()} -> {df_train['datetime'].max()}")
print(f"Reference date   : {TRAIN_MAX_DATE}")


In [ ]:
# days_since_review — age of review relative to end of train period
# Safe feature: pure timestamp, no rating information
for df_ in [df_train, df_val, df_test]:
    df_['days_since_review'] = (TRAIN_MAX_DATE - df_['datetime']).dt.days

print("days_since_review stats (train):")
print(df_train['days_since_review'].describe().round(1))


---
## Step 4 — User Features (LOO)

**user_avg_rating**: mean rating this user gives, excluding this row's own rating.
LOO formula: (sum_of_all_ratings - this_rating) / (count - 1)

**user_rating_count**: how many products this user has rated (excluding this row).
Pure count — no target leakage.

Both computed from train only. Val/test get a straight lookup.


In [ ]:
# Full user aggregates from train
user_agg = df_train.groupby('userId')['ratings'].agg(
    user_sum   = 'sum',
    user_count = 'count',
).reset_index()

# Global average — used as fallback for cold start and single-review users
global_avg = df_train['ratings'].mean()
print(f"Global avg rating : {global_avg:.4f}")
print(f"Unique users      : {len(user_agg):,}")


In [ ]:
# Merge onto train and compute LOO
df_train = df_train.merge(user_agg, on='userId', how='left')

# LOO avg: subtract this row's rating from sum and count
df_train['user_avg_rating'] = (
    (df_train['user_sum'] - df_train['ratings']) /
    (df_train['user_count'] - 1).clip(lower=1)
)
# LOO count: exclude this row
df_train['user_rating_count'] = (df_train['user_count'] - 1).clip(lower=0)

# Fill single-review users (LOO undefined) with global avg
df_train['user_avg_rating'] = df_train['user_avg_rating'].fillna(global_avg)

# Clip to valid rating range (1-5) — handles edge cases
df_train['user_avg_rating'] = df_train['user_avg_rating'].clip(1.0, 5.0)

df_train.drop(columns=['user_sum', 'user_count'], inplace=True)

print(f"user_avg_rating — mean: {df_train['user_avg_rating'].mean():.4f}  "
      f"min: {df_train['user_avg_rating'].min():.4f}  "
      f"max: {df_train['user_avg_rating'].max():.4f}")
print(f"user_rating_count — mean: {df_train['user_rating_count'].mean():.2f}  "
      f"max: {df_train['user_rating_count'].max():.0f}")


---
## Step 5 — Product Features (LOO)

**product_avg_rating**: mean rating this product receives, excluding this row.
**product_rating_count**: how many users rated this product, excluding this row.
**product_rating_std**: spread of ratings — captures polarising products.
  (Small LOO leakage for std is accepted — minor effect.)


In [ ]:
# Full product aggregates from train
product_agg = df_train.groupby('productId')['ratings'].agg(
    product_sum   = 'sum',
    product_count = 'count',
    product_std   = 'std',
).reset_index()

# std NaN for single-review products → fill with 0
product_agg['product_std'] = product_agg['product_std'].fillna(0)

# Global product stats for fallback
global_product_avg = product_agg['product_sum'].sum() / product_agg['product_count'].sum()
global_product_std = product_agg['product_std'].mean()

print(f"Global product avg : {global_product_avg:.4f}")
print(f"Global product std : {global_product_std:.4f}")
print(f"Unique products    : {len(product_agg):,}")


In [ ]:
# Merge onto train and compute LOO
df_train = df_train.merge(product_agg, on='productId', how='left')

df_train['product_avg_rating'] = (
    (df_train['product_sum'] - df_train['ratings']) /
    (df_train['product_count'] - 1).clip(lower=1)
)
df_train['product_rating_count'] = (df_train['product_count'] - 1).clip(lower=0)
df_train['product_rating_std']   = df_train['product_std']

# Fill and clip
df_train['product_avg_rating'] = (df_train['product_avg_rating']
                                   .fillna(global_product_avg)
                                   .clip(1.0, 5.0))
df_train['product_rating_std'] = df_train['product_rating_std'].fillna(0)

df_train.drop(columns=['product_sum','product_count','product_std'], inplace=True)

print(f"product_avg_rating — mean: {df_train['product_avg_rating'].mean():.4f}  "
      f"min: {df_train['product_avg_rating'].min():.4f}  "
      f"max: {df_train['product_avg_rating'].max():.4f}")
print(f"product_rating_count — mean: {df_train['product_rating_count'].mean():.2f}  "
      f"max: {df_train['product_rating_count'].max():.0f}")


---
## Step 6 — Apply Features to Val & Test

Build lookup tables from train stats.
No LOO needed for val/test — those rows were never in train aggregation.


In [ ]:
# Build lookup tables from train aggregates
user_lookup = user_agg.copy()
user_lookup['user_avg_rating']   = (user_agg['user_sum'] / user_agg['user_count']).clip(1.0, 5.0)
user_lookup['user_rating_count'] = user_agg['user_count']
user_lookup = user_lookup[['userId','user_avg_rating','user_rating_count']]

product_lookup = product_agg.copy()
product_lookup['product_avg_rating']   = (product_agg['product_sum'] / product_agg['product_count']).clip(1.0, 5.0)
product_lookup['product_rating_count'] = product_agg['product_count']
product_lookup['product_rating_std']   = product_agg['product_std']
product_lookup = product_lookup[['productId','product_avg_rating',
                                  'product_rating_count','product_rating_std']]

print(f"User lookup    : {user_lookup.shape}")
print(f"Product lookup : {product_lookup.shape}")


In [ ]:
def apply_features(df, user_lookup, product_lookup):
    df = df.merge(user_lookup,    on='userId',    how='left')
    df = df.merge(product_lookup, on='productId', how='left')
    return df

df_val  = apply_features(df_val,  user_lookup, product_lookup)
df_test = apply_features(df_test, user_lookup, product_lookup)

print(f"Val  : {df_val.shape}")
print(f"Test : {df_test.shape}")


---
## Step 7 — Cold Start Handling

Users/products in val/test not seen in train → NaN after merge.
Fill with safe global fallbacks computed from stats tables.


In [ ]:
FEATURE_COLS = [
    'user_avg_rating', 'user_rating_count',
    'product_avg_rating', 'product_rating_count', 'product_rating_std',
    'days_since_review'
]

# Count cold start cases
for name, df_ in [('Val', df_val), ('Test', df_test)]:
    cu = df_['user_avg_rating'].isna().sum()
    cp = df_['product_avg_rating'].isna().sum()
    print(f"{name} — cold start users: {cu:,}  cold start products: {cp:,}")


In [ ]:
fills = {
    'user_avg_rating'     : global_avg,          # assume average user
    'user_rating_count'   : 0,                   # no history
    'product_avg_rating'  : global_product_avg,  # assume average product
    'product_rating_count': 0,                   # no history
    'product_rating_std'  : global_product_std,  # assume average spread
}

for df_ in [df_val, df_test]:
    df_.fillna(fills, inplace=True)

# Verify no NaNs remain
for name, df_ in [('Train',df_train),('Val',df_val),('Test',df_test)]:
    nans = df_[FEATURE_COLS].isna().sum().sum()
    print(f"{name} — NaNs in features: {nans}  {'✓' if nans==0 else '✗ CHECK!'}")


---
## Step 8 — Final Feature Overview

In [ ]:
print(f"Total features : {len(FEATURE_COLS)}")
print()
print(f"{'Feature':<30} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 60)
for f in FEATURE_COLS:
    m  = df_train[f].mean()
    s  = df_train[f].std()
    mn = df_train[f].min()
    mx = df_train[f].max()
    print(f"{f:<30} {m:>8.3f} {s:>8.3f} {mn:>8.3f} {mx:>8.3f}")


---
## Step 9 — Save Enriched CSVs

In [ ]:
train_out = os.path.join(DATA_DIR, 'train_features.csv')
val_out   = os.path.join(DATA_DIR, 'val_features.csv')
test_out  = os.path.join(DATA_DIR, 'test_features.csv')

df_train.to_csv(train_out, index=False)
df_val.to_csv(val_out,     index=False)
df_test.to_csv(test_out,   index=False)

for fpath in [train_out, val_out, test_out]:
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  {os.path.basename(fpath):<25}  {size_mb:.1f} MB")


In [ ]:
# Reload and confirm
train_check = pd.read_csv(train_out)
val_check   = pd.read_csv(val_out)
test_check  = pd.read_csv(test_out)

print("Reloaded shapes:")
print(f"  train_features : {train_check.shape}")
print(f"  val_features   : {val_check.shape}")
print(f"  test_features  : {test_check.shape}")

for name, df_ in [('train',train_check),('val',val_check),('test',test_check)]:
    nans = df_[FEATURE_COLS].isna().sum().sum()
    print(f"  {name} NaNs: {nans}  {'✓' if nans==0 else '✗'}")

print()
print("=" * 55)
print("FEATURE ENGINEERING COMPLETE (v4 — simplified)")
print("=" * 55)
print(f"  Features : {FEATURE_COLS}")
print(f"  Train    : {len(train_check):,} rows")
print(f"  Val      : {len(val_check):,} rows")
print(f"  Test     : {len(test_check):,} rows")
print("=" * 55)
